In [ ]:
"""
Notebook-style training script for a custom fine-tuned LLM.

This is designed to run in Google Colab or a GPU notebook.
It keeps everything in one file for easy copy/paste execution.
"""

In [ ]:
from __future__ import annotations

In [ ]:
import json
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

In [ ]:
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

=============================================================================
Configuration
=============================================================================

In [ ]:
# Base model to fine-tune (small, fast, decent for style tuning).
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
# Where your dataset lives (JSONL or CSV).
TRAIN_DATA_PATH = "data/train.jsonl"
VALID_DATA_PATH = "data/valid.jsonl"

In [ ]:
# Output directory for the LoRA adapter or full model.
OUTPUT_DIR = "finetuned-adapter"

In [ ]:
# Training parameters
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 2
WARMUP_RATIO = 0.05

In [ ]:
# Set to True to enable 4-bit loading (requires bitsandbytes).
USE_4BIT = True

=============================================================================
Data Format
=============================================================================
Each JSONL row should be a dict with either:
1) {"prompt": "...", "completion": "..."}
OR
2) {"input": "...", "output": "..."}

The script normalizes both into a single training string.

In [ ]:
@dataclass
class Example:
    prompt: str
    completion: str

In [ ]:
SYSTEM_PROMPT = (
    "You are a marketing copywriter who writes in Adriana James' voice. "
    "Tone: warm, empowering, clear, concise, action-oriented. "
    "Avoid hype and fake urgency. Use confident, supportive language."
)

In [ ]:
def normalize_example(row: Dict) -> Example:
    if "prompt" in row and "completion" in row:
        return Example(prompt=row["prompt"], completion=row["completion"])
    if "input" in row and "output" in row:
        return Example(prompt=row["input"], completion=row["output"])
    raise ValueError("Each row must contain prompt/completion or input/output.")

In [ ]:
def format_example(example: Example) -> str:
    # Simple instruction format suitable for causal LM fine-tuning.
    return (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n[/SYSTEM]\n"
        f"[USER]\n{example.prompt}\n[/USER]\n"
        f"[ASSISTANT]\n{example.completion}\n</s>"
    )

In [ ]:
def load_jsonl(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

In [ ]:
def build_dataset(train_path: str, valid_path: str) -> Tuple[Dataset, Dataset]:
    train_rows = load_jsonl(train_path)
    valid_rows = load_jsonl(valid_path) if os.path.exists(valid_path) else []

    if not valid_rows:
        random.shuffle(train_rows)
        split_idx = max(1, int(len(train_rows) * 0.9))
        valid_rows = train_rows[split_idx:]
        train_rows = train_rows[:split_idx]

    def to_text(row: Dict) -> Dict:
        ex = normalize_example(row)
        return {"text": format_example(ex)}

    train_ds = Dataset.from_list([to_text(r) for r in train_rows])
    valid_ds = Dataset.from_list([to_text(r) for r in valid_rows])
    return train_ds, valid_ds

=============================================================================
Tokenization
=============================================================================

In [ ]:
def tokenize_dataset(tokenizer, dataset: Dataset) -> Dataset:
    def tokenize_fn(batch: Dict) -> Dict:
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

    return dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

=============================================================================
Model Setup
=============================================================================

In [ ]:
def load_model_and_tokenizer() -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {}
    if USE_4BIT:
        model_kwargs.update(
            dict(
                load_in_4bit=True,
                torch_dtype=torch.float16,
                device_map="auto",
            )
        )

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

    if USE_4BIT:
        model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer

=============================================================================
Training
=============================================================================

In [ ]:
def train():
    train_ds, valid_ds = build_dataset(TRAIN_DATA_PATH, VALID_DATA_PATH)
    model, tokenizer = load_model_and_tokenizer()

    tokenized_train = tokenize_dataset(tokenizer, train_ds)
    tokenized_valid = tokenize_dataset(tokenizer, valid_ds)

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False
    )

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=10,
        evaluation_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print(f"Saved adapter/model to: {OUTPUT_DIR}")

=============================================================================
Quick Evaluation
=============================================================================

In [ ]:
def generate_sample(prompt: str):
    model, tokenizer = load_model_and_tokenizer()
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    text = format_example(Example(prompt=prompt, completion=""))
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    print(decoded)

In [ ]:
if __name__ == "__main__":
    # Step 1: Train
    train()

    # Step 2: Quick sanity check
    generate_sample("Write a short welcome email for new subscribers.")